# 04 Komparasi Model & Penyimpanan Artefak


---
### 1. Setup Environment


In [7]:
import sys, json, joblib
from pathlib import Path

ROOT_DIR = Path(r"c:\Project\napasloka\napasloka")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))


---
### 2. Parameter Eksperimen & Pembentukan Data


In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

from src.features.lag_features import create_lag_features
from src.features.scenarios import resolve_scenario
from src.data.clean_data import chronological_train_test_split
from src.models.estimators import create_estimator
from src.models.estimators import tune_on_training_set
from src.utils.metrics import calculate_regression_metrics
from src.models.interpretation import calculate_tree_shap, mean_absolute_shap
from src.models.interpretation import gain_feature_importance
from src.utils.config import load_yaml_config

warnings.filterwarnings('ignore')

STATION_ID = "pantai-makmur"
POLLUTANT = "co2"
SCENARIO = "scenario_2"
SAVE_ARTIFACTS = True

# Pemuatan data stasiun
df_raw = pd.read_csv(ROOT_DIR / "data" / "processed" / f"jabodetabek_{POLLUTANT}_dataset.csv")
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], utc=True).dt.tz_convert("Asia/Jakarta")

coord_id = "-6.1_107.0"
df_station = df_raw[(df_raw["location"] == coord_id) | (df_raw["location"] == STATION_ID)].sort_values("datetime").reset_index(drop=True)
df_feat = create_lag_features(df_station, pollutant=POLLUTANT, scenario=SCENARIO, drop_incomplete_rows=True)

scenario_def = resolve_scenario(POLLUTANT, SCENARIO)
feature_cols = list(scenario_def.feature_names)
target_col = scenario_def.pollutant_column

train_df, test_df = chronological_train_test_split(df_feat, test_fraction=0.2)
x_train, y_train = train_df[feature_cols], train_df[target_col]
x_test, y_test = test_df[feature_cols], test_df[target_col]

print(f"Data Latih: {len(x_train):,} sampel | Data Uji: {len(x_test):,} sampel | Fitur: {len(feature_cols)} kolom")


Data Latih: 12,433 sampel | Data Uji: 3,109 sampel | Fitur: 15 kolom


---
### 3. Hyperparameter Tuning pada Data Latih


In [9]:
models = ["xgboost", "lightgbm"]
tuned_estimators = {}
best_params = {}

for algo in models:
    print(f"Tuning {algo.upper()} dengan TimeSeriesSplit(3)...")
    base_est = create_estimator(algo, parameters={"random_state": 42})
    search = tune_on_training_set(base_est, algo, x_train, y_train)
    tuned_estimators[algo] = search.best_estimator_
    best_params[algo] = search.best_params_
    print(f"✓ Best params {algo.upper()}: {search.best_params_}")


Tuning XGBOOST dengan TimeSeriesSplit(3)...
✓ Best params XGBOOST: {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05}
Tuning LIGHTGBM dengan TimeSeriesSplit(3)...
✓ Best params LIGHTGBM: {'n_estimators': 500, 'max_depth': 9, 'learning_rate': 0.05}


---
### 4. Pelatihan Model & Evaluasi Kinerja (Data Uji)


In [10]:
results = {}

for algo in models:
    model = tuned_estimators[algo]
    y_pred = model.predict(x_test)
    metrics = calculate_regression_metrics(y_test, y_pred)
    results[algo] = {"model": model, "y_pred": y_pred, "metrics": metrics}

# Tabel Perbandingan Kinerja
df_comp = pd.DataFrame({
    algo.upper(): {
        "MAE (ppm)": results[algo]["metrics"]["mae"],
        "RMSE (ppm)": results[algo]["metrics"]["rmse"],
        "MAPE (%)": results[algo]["metrics"]["mape"],
        "R² Score": results[algo]["metrics"]["r2"],
    }
    for algo in models
}).T

print("=== HASIL KOMPARASI METRIK EVALUASI (DATA UJI) ===")
display(df_comp)


=== HASIL KOMPARASI METRIK EVALUASI (DATA UJI) ===


,MAE (ppm),RMSE (ppm),MAPE (%),R² Score
XGBOOST,1.269834,1.825504,0.276422,0.978932
LIGHTGBM,1.229809,1.765211,0.267752,0.980301


---
### 5. Simpan Model & Artefak ke `artifacts/experiments/`


In [11]:
if SAVE_ARTIFACTS:
    for algo in models:
        artifact_dir = ROOT_DIR / "artifacts" / "experiments" / STATION_ID / POLLUTANT / SCENARIO / algo
        artifact_dir.mkdir(parents=True, exist_ok=True)
        
        joblib.dump(results[algo]["model"], artifact_dir / "model.joblib")
        with (artifact_dir / "best_params.json").open("w", encoding="utf-8") as f:
            json.dump(best_params[algo], f, indent=2)
        with (artifact_dir / "metrics.json").open("w", encoding="utf-8") as f:
            json.dump(results[algo]["metrics"], f, indent=2)
        pd.DataFrame({
            "datetime": test_df["datetime"].astype(str),
            "actual": y_test.values,
            "predicted": results[algo]["y_pred"],
            "residual": y_test.values - results[algo]["y_pred"],
        }).to_csv(artifact_dir / "predictions.csv", index=False)
        
        print(f"✓ Berhasil menyimpan artefak {algo.upper()} ke: {artifact_dir}")


✓ Berhasil menyimpan artefak XGBOOST ke: c:\Project\napasloka\napasloka\artifacts\experiments\pantai-makmur\co2\scenario_2\xgboost
✓ Berhasil menyimpan artefak LIGHTGBM ke: c:\Project\napasloka\napasloka\artifacts\experiments\pantai-makmur\co2\scenario_2\lightgbm


---
### 6. Batch Experiment Runner: Matriks 120 Model & Registri Model Terbaik

In [12]:
from src.models.experiments import build_experiment_matrix
from src.models.experiments import ExperimentRunner
from src.models.experiments import execute_single_experiment
from src.models.experiments import select_best_models

matrix_specs = build_experiment_matrix()
print(f"Total Matriks Eksperimen Penelitian: {len(matrix_specs)} kombinasi")

runner = ExperimentRunner()

selected_specs = [s for s in matrix_specs if s.location == STATION_ID]
print(f"Memeriksa status {len(selected_specs)} eksperimen untuk stasiun '{STATION_ID}'...\n")

for idx, spec in enumerate(selected_specs, start=1):
    exp_id = spec.experiment_id
    sudah_selesai = not runner.registry.should_run(spec)
    status_text = "✓ SELESAI (Tersimpan di artifacts/)" if sudah_selesai else "○ BELUM DILATIH"
    print(f"[{idx}/{len(selected_specs)}] {exp_id:<55} -> {status_text}")


Total Matriks Eksperimen Penelitian: 120 kombinasi
Memeriksa status 8 eksperimen untuk stasiun 'pantai-makmur'...

[1/8] pantai-makmur__co__scenario_1__xgboost                  -> ✓ SELESAI (Tersimpan di artifacts/)
[2/8] pantai-makmur__co__scenario_1__lightgbm                 -> ✓ SELESAI (Tersimpan di artifacts/)
[3/8] pantai-makmur__co__scenario_2__xgboost                  -> ✓ SELESAI (Tersimpan di artifacts/)
[4/8] pantai-makmur__co__scenario_2__lightgbm                 -> ✓ SELESAI (Tersimpan di artifacts/)
[5/8] pantai-makmur__co2__scenario_1__xgboost                 -> ✓ SELESAI (Tersimpan di artifacts/)
[6/8] pantai-makmur__co2__scenario_1__lightgbm                -> ✓ SELESAI (Tersimpan di artifacts/)
[7/8] pantai-makmur__co2__scenario_2__xgboost                 -> ✓ SELESAI (Tersimpan di artifacts/)
[8/8] pantai-makmur__co2__scenario_2__lightgbm                -> ✓ SELESAI (Tersimpan di artifacts/)


In [ ]:
best_registry = select_best_models()
print(f"✓ Registri model terbaik diperbarui: {best_registry['total_combinations']} kombinasi terbaik tersimpan di artifacts/registry/best_models.json\n")

summary_path = ROOT_DIR / "artifacts" / "summary_metrics.csv"
if summary_path.exists():
    df_summary = pd.read_csv(summary_path)
    print(f"=== REKAPITULASI 120 MODEL EKSPERIMEN (Total: {len(df_summary)} Model) ===")
    display(df_summary[["location", "pollutant", "scenario", "algorithm", "mae", "rmse", "mape", "r2"]].head(10))
